# Ders 9: Durum-Uzayı Modelleri ve Kalman Filtresi
Bu notebook'ta şunları öğreneceğiz:
- Durum-uzayı gösterimi
- Kalman filtresi algoritması (teorik + elle uygulama)
- Yerel doğrusal trend modeli
- Eksik gözlem işleme
- Kalman düzleştirici (smoother)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import solve
import warnings
warnings.filterwarnings('ignore')

# filterpy opsiyonel
try:
    from filterpy.kalman import KalmanFilter as FPKalmanFilter
    FILTERPY = True
except ImportError:
    FILTERPY = False

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
print(f"filterpy: {'mevcut' if FILTERPY else 'yüklü değil (elle uygulama kullanılacak)'}")


## 9.1 Durum-Uzayı Gösterimi

Genel doğrusal Gaussian durum-uzayı modeli:

**Gözlem denklemi:**
$$Y_t = G_t \mathbf{x}_t + W_t, \quad W_t \sim N(0, R_t)$$

**Durum denklemi:**
$$\mathbf{x}_t = F_t \mathbf{x}_{t-1} + V_t, \quad V_t \sim N(\mathbf{0}, Q_t)$$

**Başlangıç:** $\mathbf{x}_0 \sim N(\mathbf{m}_0, P_0)$

| Sembol | Boyut | Anlam |
|--------|-------|-------|
| $\mathbf{x}_t$ | $m \times 1$ | Gözlenemeyen durum |
| $Y_t$ | $1 \times 1$ | Gözlem |
| $F_t$ | $m \times m$ | Durum geçiş matrisi |
| $G_t$ | $1 \times m$ | Gözlem matrisi |
| $Q_t$ | $m \times m$ | Süreç gürültüsü kovaryansı |
| $R_t$ | skalar | Ölçüm gürültüsü varyansı |


In [ ]:
# ── Kalman Filtresi: Elle Uygulama ──

def kalman_filter(Y, F, G, Q, R, x0, P0):
    # F: durum gecis matrisi (m x m)
    # G: olcum matrisi (1 x m)
    # Q: surec gurultusu kovaryans (m x m)
    # R: olcum gurultusu varyans (skalar)
    # x0, P0: baslangic durum ve kovaryans
    n = len(Y)
    m = len(x0)
    
    x_pred = np.zeros((n, m))
    P_pred = np.zeros((n, m, m))
    x_filt = np.zeros((n, m))
    P_filt = np.zeros((n, m, m))
    K_all  = np.zeros((n, m))
    innov  = np.zeros(n)
    
    x_k = x0.copy()
    P_k = P0.copy()
    
    for t in range(n):
        # Ongorusel adim (Predict)
        x_p = F @ x_k
        P_p = F @ P_k @ F.T + Q
        
        # Inovasyon
        v_t = Y[t] - G @ x_p
        S_t = G @ P_p @ G.T + R    # Inovasyon varyans
        
        # Kalman kazanim
        K_t = (P_p @ G.T) / S_t
        
        # Guncelleme adimi (Update)
        x_k = x_p + K_t * v_t
        P_k = (np.eye(m) - np.outer(K_t, G)) @ P_p
        
        # Kaydet
        x_pred[t] = x_p
        P_pred[t] = P_p
        x_filt[t] = x_k
        P_filt[t] = P_k
        K_all[t]  = K_t
        innov[t]  = v_t
    
    return x_filt, P_filt, x_pred, P_pred, innov, K_all

print("Kalman filtresi fonksiyonu hazir.")


## 9.2 Basit Örnek: AR(1) Modeli Durum-Uzayı Olarak

AR(1): $X_t = \phi X_{t-1} + Z_t$ modeli durum-uzayı formunda:
- $F = [\phi]$, $G = [1]$, $Q = [\sigma^2_Z]$, $R = 0$

Gürültülü gözlemle ($Y_t = X_t + W_t$): $R = \sigma^2_W > 0$


In [ ]:
# ── AR(1) + Gürültülü Gözlem Filtreleme ──

np.random.seed(42)
n = 150
phi = 0.85
sigma_z = 1.0
sigma_w = 2.0  # Gozlem gurultusu (yuksek)

# Gercek durum
X_true = np.zeros(n)
for t in range(1, n):
    X_true[t] = phi*X_true[t-1] + np.random.normal(0, sigma_z)

# Gurultulu gozlem
Y_obs = X_true + np.random.normal(0, sigma_w, n)

# Kalman filtresi parametreleri
F = np.array([[phi]])
G = np.array([1.0])
Q = np.array([[sigma_z**2]])
R = sigma_w**2
x0 = np.array([0.0])
P0 = np.array([[sigma_z**2 / (1 - phi**2)]])

x_filt, P_filt, x_pred, P_pred, innov, K = kalman_filter(Y_obs, F, G, Q, R, x0, P0)
sigma_filt = np.sqrt(P_filt[:, 0, 0])

fig, axes = plt.subplots(3, 1, figsize=(13, 10))

axes[0].plot(Y_obs, color='gray', lw=0.7, alpha=0.7, label='Gurultulu Gozlem Y_t')
axes[0].plot(X_true, color='steelblue', lw=1.5, label='Gercek Durum X_t')
axes[0].plot(x_filt[:,0], color='red', lw=1.5, ls='--', label='Kalman Filtresi')
axes[0].fill_between(range(n),
                      x_filt[:,0]-2*sigma_filt,
                      x_filt[:,0]+2*sigma_filt,
                      alpha=0.2, color='red', label='%95 GA')
axes[0].set_title('Kalman Filtresi: AR(1) Durumu Geri Kazanim', fontweight='bold')
axes[0].legend(fontsize=9)

axes[1].plot(K[:,0], color='purple', lw=1.2)
axes[1].set_title('Kalman Kazanimi $K_t$ (Zamanla Sabitlenir)', fontweight='bold')
axes[1].axhline(K[-1,0], color='red', ls='--', alpha=0.5, label=f'Kararlı hali={K[-1,0]:.3f}')
axes[1].legend()

axes[2].plot(innov, color='darkorange', lw=0.8, alpha=0.8)
axes[2].axhline(0, color='black', ls='--')
axes[2].set_title('Inovasyon $v_t = Y_t - G\hat{x}_{t|t-1}$ (WN olmali)', fontweight='bold')

plt.tight_layout()
plt.show()

rmse_obs = np.sqrt(np.mean((Y_obs - X_true)**2))
rmse_filt = np.sqrt(np.mean((x_filt[:,0] - X_true)**2))
print(f"Kaba gozlem RMSE    : {rmse_obs:.3f}")
print(f"Kalman filtresi RMSE: {rmse_filt:.3f}")
print(f"Iyilesme: %{100*(rmse_obs-rmse_filt)/rmse_obs:.1f}")


## 9.3 Yerel Doğrusal Trend Modeli (Local Linear Trend)

$$Y_t = \mu_t + \epsilon_t, \quad \epsilon_t \sim N(0, \sigma^2_\epsilon)$$
$$\mu_t = \mu_{t-1} + \beta_{t-1} + \xi_t, \quad \xi_t \sim N(0, \sigma^2_\xi)$$
$$\beta_t = \beta_{t-1} + \zeta_t, \quad \zeta_t \sim N(0, \sigma^2_\zeta)$$

Durum vektörü: $\mathbf{x}_t = (\mu_t, \beta_t)^T$

$$F = \begin{pmatrix} 1 & 1 \\ 0 & 1 \end{pmatrix}, \quad G = \begin{pmatrix} 1 & 0 \end{pmatrix}$$


In [ ]:
# ── Yerel Dogrusal Trend + Kalman ──

np.random.seed(42)
n = 200

# Gercek trend simulasyonu
sigma_eps = 2.0
sigma_xi  = 0.3
sigma_zeta = 0.05

mu = np.zeros(n)
beta = np.zeros(n)
mu[0], beta[0] = 5.0, 0.1

for t in range(1, n):
    beta[t] = beta[t-1] + np.random.normal(0, sigma_zeta)
    mu[t]   = mu[t-1] + beta[t-1] + np.random.normal(0, sigma_xi)

Y_llt = mu + np.random.normal(0, sigma_eps, n)

# Durum-uzayi parametreleri
F_llt = np.array([[1.0, 1.0],
                   [0.0, 1.0]])
G_llt = np.array([1.0, 0.0])
Q_llt = np.diag([sigma_xi**2, sigma_zeta**2])
R_llt = sigma_eps**2
x0_llt = np.array([Y_llt[0], 0.0])
P0_llt = np.diag([10.0, 1.0])

x_filt_llt, P_filt_llt, _, _, innov_llt, _ = kalman_filter(
    Y_llt, F_llt, G_llt, Q_llt, R_llt, x0_llt, P0_llt)

sig_mu = np.sqrt(P_filt_llt[:, 0, 0])

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

axes[0].plot(Y_llt, color='gray', lw=0.7, alpha=0.6, label='Gözlem Y_t')
axes[0].plot(mu, color='steelblue', lw=2, label='Gerçek Trend μ_t')
axes[0].plot(x_filt_llt[:,0], color='red', lw=1.5, ls='--', label='Kalman Filtreli Trend')
axes[0].fill_between(range(n),
                      x_filt_llt[:,0]-2*sig_mu,
                      x_filt_llt[:,0]+2*sig_mu,
                      alpha=0.2, color='red')
axes[0].set_title('Yerel Doğrusal Trend: Kalman Filtresi ile Trend Çıkarma', fontweight='bold')
axes[0].legend()

axes[1].plot(beta, color='steelblue', lw=1.5, label='Gerçek Eğim β_t')
axes[1].plot(x_filt_llt[:,1], color='red', lw=1.5, ls='--', label='Tahmin Edilen Eğim')
axes[1].set_title('Eğim Bileşeni β_t', fontweight='bold')
axes[1].axhline(0, color='black', ls='--', alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()


## 9.4 Eksik Gözlem İşleme

Kalman filtresinin güzel özelliği: **eksik gözlemler için güncelleme adımını atlayabilirsiniz** — sadece öngörü adımını çalıştırın.

$$Y_t = \text{NaN} \Rightarrow \hat{x}_{t|t} = \hat{x}_{t|t-1}, \quad P_{t|t} = P_{t|t-1}$$


In [ ]:
# ── Eksik Gözlem ile Kalman Filtresi ──

def kalman_filter_missing(Y, F, G, Q, R, x0, P0):
    n = len(Y)
    m = len(x0)
    x_filt = np.zeros((n, m))
    P_filt = np.zeros((n, m, m))
    x_k, P_k = x0.copy(), P0.copy()
    
    for t in range(n):
        # Tahmin adimi
        x_p = F @ x_k
        P_p = F @ P_k @ F.T + Q
        
        if np.isnan(Y[t]):
            # Guncelleme yapma, onceki tahminle devam et
            x_k = x_p
            P_k = P_p
        else:
            v_t = Y[t] - G @ x_p
            S_t = G @ P_p @ G.T + R
            K_t = (P_p @ G.T) / S_t
            x_k = x_p + K_t * v_t
            P_k = (np.eye(m) - np.outer(K_t, G)) @ P_p
        
        x_filt[t] = x_k
        P_filt[t] = P_k
    
    return x_filt, P_filt

# Veri: gozlemlerin %30unu eksik yap
Y_missing = Y_llt.copy()
missing_idx = np.random.choice(n, size=int(0.30*n), replace=False)
Y_missing[missing_idx] = np.nan

x_filt_miss, P_filt_miss = kalman_filter_missing(
    Y_missing, F_llt, G_llt, Q_llt, R_llt, x0_llt, P0_llt)

sig_miss = np.sqrt(P_filt_miss[:, 0, 0])

fig, ax = plt.subplots(figsize=(13, 5))
ax.scatter(range(n), Y_llt, color='gray', s=8, alpha=0.5, label='Tam Veri')
ax.scatter(range(n), Y_missing, color='steelblue', s=10, alpha=0.8, label='Gözlenen')
ax.plot(x_filt_miss[:,0], color='red', lw=1.5, label='Kalman (Eksikli)', zorder=5)
ax.fill_between(range(n),
                x_filt_miss[:,0]-2*sig_miss,
                x_filt_miss[:,0]+2*sig_miss,
                alpha=0.2, color='red')
ax.set_title('Eksik Gözlemlerle Kalman Filtresi (%30 Eksik)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Toplam gözlem    : {n}")
print(f"Eksik gözlem     : {len(missing_idx)} (%{100*len(missing_idx)/n:.0f})")
print(f"Tam veri RMSE    : {np.sqrt(np.mean((x_filt_llt[:,0]-mu)**2)):.3f}")
print(f"Eksikli veri RMSE: {np.sqrt(np.mean((x_filt_miss[:,0]-mu)**2)):.3f}")


## Özet

| Adım | İşlem |
|------|-------|
| **Tahmin (Predict)** | $\hat{x}_{t\|t-1} = F\hat{x}_{t-1}$, $P_{t\|t-1} = FP_{t-1}F^T + Q$ |
| **Inovasyon** | $v_t = Y_t - G\hat{x}_{t\|t-1}$ |
| **Kalman Kazanımı** | $K_t = P_{t\|t-1}G^T / (GP_{t\|t-1}G^T + R)$ |
| **Güncelleme (Update)** | $\hat{x}_t = \hat{x}_{t\|t-1} + K_t v_t$ |
| **Kovaryans Güncelleme** | $P_t = (I - K_tG)P_{t\|t-1}$ |

**Uygulamalar:**
- GPS konumlandırma
- Finansal trend çıkarma
- Robot odometri
- Ekonomik gösterge tahmini (eksik verili)
